In [1]:
from tqdm import tqdm
import numpy as np
import warnings
from scipy.stats import ConstantInputWarning

from analyses.response_window_analysis import run_permutation_anova_by_window, run_glm_by_window
from analyses.spike_count import extract_spike_counts_from_windows

warnings.simplefilter("ignore", ConstantInputWarning)
from analyses.response_window_finder.threshold_window_detection import compute_timebinned_spikecount_per_neuron, z_score, \
    threshold_and_fill_gap, extract_consecutive_ranges, remove_consecutive_tuples, \
    find_corresponding_values_for_index_ranges
%load_ext autoreload
%autoreload 2

In [2]:
date = "2023-09-26"
round_no = 1
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Zombies'
results = []
zombies_timebin_spikecount_list= compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, 'Zombies')
for _, r in tqdm(zombies_timebin_spikecount_list.iterrows(), total=len(zombies_timebin_spikecount_list), desc="Processing each neuron"):
    data = r['TotalSpikeCountList']
    neuron = r['NeuronID']
    normalized_data = z_score(data)
    thresh = 0.5
    change_points = threshold_and_fill_gap(normalized_data, thresh)
    windows = extract_consecutive_ranges(change_points)
    filtered_windows = remove_consecutive_tuples(windows)
    time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
    if len(time_windows) > 0:
        print(f"---------------- {neuron} ----------------")
        print(time_windows)
        for start_time, end_time in time_windows:
            results.append({
                'NeuronID': neuron,
                'WindowStart_ms': int(start_time * 1000),
                'WindowEnd_ms': int(end_time * 1000)
            })



Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


Processing each neuron: 100%|██████████| 32/32 [00:00<00:00, 8238.77it/s]

---------------- 2023-09-26_1_Channel.C_003 ----------------
[(0.65, 0.8)]
---------------- 2023-09-26_1_Channel.C_004 ----------------
[(0.05, 0.35), (0.65, 1.0)]
---------------- 2023-09-26_1_Channel.C_005_Unit 1 ----------------
[(0.3, 0.4), (0.7, 0.9)]
---------------- 2023-09-26_1_Channel.C_006 ----------------
[(0.65, 0.85)]
---------------- 2023-09-26_1_Channel.C_010 ----------------
[(0.65, 0.8)]
---------------- 2023-09-26_1_Channel.C_011_Unit 1 ----------------
[(0.35, 0.5)]
---------------- 2023-09-26_1_Channel.C_012_Unit 3 ----------------
[(0.35, 0.65), (0.75, 0.85), (1.7, 1.9)]
---------------- 2023-09-26_1_Channel.C_014_Unit 1 ----------------
[(0.15, 0.5)]
---------------- 2023-09-26_1_Channel.C_017_Unit 1 ----------------
[(0.15, 0.25), (0.35, 0.5)]
---------------- 2023-09-26_1_Channel.C_018_Unit 1 ----------------
[(0.05, 0.15), (1.75, 1.85)]
---------------- 2023-09-26_1_Channel.C_018_Unit 2 ----------------
[(0.1, 0.2), (0.35, 0.45), (0.7, 0.8)]
---------------- 20

In [3]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['NeuronID'])
results_df.head()

,NeuronID,WindowStart_ms,WindowEnd_ms
0,2023-09-26_1_Channel.C_003,650,800
1,2023-09-26_1_Channel.C_004,50,350
2,2023-09-26_1_Channel.C_004,650,1000
3,2023-09-26_1_Channel.C_005_Unit 1,300,400
4,2023-09-26_1_Channel.C_005_Unit 1,700,900


In [6]:
final_df = extract_spike_counts_from_windows(results_df)
zombies_df = final_df[final_df['MonkeyGroup'] == 'Zombies']
run_permutation_anova_by_window(zombies_df,
                                    category_col='MonkeyName',
                                    neuron_col='NeuronID',
                                    count_col='SpikeCount',
                                    window_start_col='WindowStart_ms',
                                    window_end_col='WindowEnd_ms',
                                    n_permutations=1000,
                                    alpha=0.05,
                                    plot=False)

Extracting spike counts:   0%|          | 0/35 [00:00<?, ?it/s]


Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


Running Perm ANOVA per (Neuron, Window): 100%|██████████| 35/35 [00:04<00:00,  8.18it/s]


All Results:
                             NeuronID  WindowStart_ms  WindowEnd_ms  \
0          2023-09-26_1_Channel.C_003             650           800   
1          2023-09-26_1_Channel.C_004              50           350   
2          2023-09-26_1_Channel.C_004             650          1000   
3   2023-09-26_1_Channel.C_005_Unit 1             300           400   
4   2023-09-26_1_Channel.C_005_Unit 1             700           900   
5          2023-09-26_1_Channel.C_006             650           850   
6          2023-09-26_1_Channel.C_010             650           800   
7   2023-09-26_1_Channel.C_011_Unit 1             350           500   
8   2023-09-26_1_Channel.C_012_Unit 3             350           650   
9   2023-09-26_1_Channel.C_012_Unit 3             750           850   
10  2023-09-26_1_Channel.C_012_Unit 3            1700          1900   
11  2023-09-26_1_Channel.C_014_Unit 1             150           500   
12  2023-09-26_1_Channel.C_017_Unit 1             150          

,NeuronID,WindowStart_ms,WindowEnd_ms,F-statistic,p-value
0,2023-09-26_1_Channel.C_003,650,800,1.303195,0.145
1,2023-09-26_1_Channel.C_004,50,350,1.741725,0.090
2,2023-09-26_1_Channel.C_004,650,1000,1.550956,0.160
3,2023-09-26_1_Channel.C_005_Unit 1,300,400,0.815823,0.493
4,2023-09-26_1_Channel.C_005_Unit 1,700,900,1.006339,0.412
5,2023-09-26_1_Channel.C_006,650,850,1.413942,0.030
6,2023-09-26_1_Channel.C_010,650,800,1.532373,0.032
7,2023-09-26_1_Channel.C_011_Unit 1,350,500,1.029580,0.394
8,2023-09-26_1_Channel.C_012_Unit 3,350,650,0.782943,0.624
9,2023-09-26_1_Channel.C_012_Unit 3,750,850,0.796947,0.556


In [8]:
window_glm_results = run_glm_by_window(zombies_df, formula = "SpikeCount ~ C(MonkeyName)")
print(window_glm_results.head())

Running GLM per (Neuron, Window): 100%|██████████| 35/35 [00:00<00:00, 188.15it/s]

                   index         Coef.      Std.Err.             z     P>|z|  \
0              Intercept -2.498002e-16      0.333333 -7.494005e-16  1.000000   
1  C(MonkeyName)[T.143H] -2.482138e+01  47070.271234 -5.273260e-04  0.999579   
2  C(MonkeyName)[T.151J] -2.482138e+01  49616.422394 -5.002654e-04  0.999601   
3   C(MonkeyName)[T.67G] -2.302585e+00      1.054093 -2.184424e+00  0.028931   
4   C(MonkeyName)[T.69X] -2.482138e+01  49616.422393 -5.002654e-04  0.999601   

         [0.025        0.975]                    NeuronID  WindowStart_ms  \
0     -0.653321      0.653321  2023-09-26_1_Channel.C_003             650   
1 -92280.857740  92231.214982  2023-09-26_1_Channel.C_003             650   
2 -97271.222312  97221.579554  2023-09-26_1_Channel.C_003             650   
3     -4.368569     -0.236602  2023-09-26_1_Channel.C_003             650   
4 -97271.222312  97221.579554  2023-09-26_1_Channel.C_003             650   

   WindowEnd_ms  
0           800  
1           800  
2 

In [9]:
from analyses.response_window_analysis import get_significant_windows, correct_glm_by_window_pvalues

# 2. Find significant windows (raw p < 0.05)
significant_windows = get_significant_windows(window_glm_results)

# 3. Correct for multiple comparisons across neurons
corrected_neuron_summary = correct_glm_by_window_pvalues(window_glm_results)

In [11]:
significant_windows[['index', 'Coef.', 'P>|z|','NeuronID', 'WindowStart_ms', 'WindowEnd_ms' ]]

,index,Coef.,P>|z|,NeuronID,WindowStart_ms,WindowEnd_ms
3,C(MonkeyName)[T.67G],-2.302585,2.893110e-02,2023-09-26_1_Channel.C_003,650,800
9,Intercept,2.209495,4.707408e-89,2023-09-26_1_Channel.C_004,50,350
10,C(MonkeyName)[T.143H],-0.523096,2.837613e-03,2023-09-26_1_Channel.C_004,50,350
18,Intercept,2.793208,2.121201e-251,2023-09-26_1_Channel.C_004,650,1000
19,C(MonkeyName)[T.143H],-0.778305,4.138833e-08,2023-09-26_1_Channel.C_004,650,1000
...,...,...,...,...,...,...
287,C(MonkeyName)[T.94B],0.538997,1.925354e-02,2023-09-26_1_Channel.C_027_Unit 1,150,250
288,Intercept,0.510826,4.788130e-02,2023-09-26_1_Channel.C_027_Unit 1,950,1050
297,Intercept,0.980829,1.547033e-06,2023-09-26_1_Channel.C_028,600,800
298,C(MonkeyName)[T.143H],-3.283414,1.295043e-03,2023-09-26_1_Channel.C_028,600,800
